# Parte 1 – Scraping de decretos de emergencia

**Grupo 11 · Temporada 2024** (1 de enero al 31 de mayo de 2024)

Pregunta del trabajo: *¿El Estado declara emergencia donde más llueve?*

En este notebook reunimos los decretos supremos de la PCM que declaran (o prorrogan) el Estado de Emergencia por lluvias, publicados en nuestra temporada, e identificamos qué departamentos menciona cada uno.

## 0. Librerías

In [1]:
import re
import time
import calendar
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# gob.pe no envía la cadena completa de certificados SSL: Chrome la completa solo,
# pero `requests` falla con CERTIFICATE_VERIFY_FAILED. `truststore` hace que Python
# use el almacén de certificados del sistema operativo (igual que el navegador).
# Así NO tenemos que desactivar la verificación con verify=False.
import truststore
truststore.inject_into_ssl()

HEADERS = {"User-Agent": "Mozilla/5.0"}
CARPETA_DATOS = Path("datos")
CARPETA_DATOS.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 120)

## 1. `robots.txt`

In [2]:
robots = requests.get("https://www.gob.pe/robots.txt", headers=HEADERS)
print(robots.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



**¿Qué prohíbe?**

- Para **todos** los bots (`User-agent: *`) solo prohíbe tres cosas: la zona de administración (`/admin/`) y cualquier URL con el parámetro `sheet` (`/*?sheet=` y `/*&sheet=`).
- A `AhrefsBot` le prohíbe **todo** el sitio (`Disallow: /`).
- A `GPTBot` y `OAI-SearchBot` los deja entrar (con las mismas restricciones que a todos), pero les pide un `Crawl-delay` de 5 y 2 segundos respectivamente.

**¿Está permitida `/busquedas`?** Sí. No aparece en ningún `Disallow` para `User-agent: *`, y nuestras URLs de búsqueda no usan el parámetro `sheet`. Las páginas de cada norma (`/institucion/pcm/normas-legales/...`) tampoco están prohibidas.

**¿Por qué igual hacemos pausas?** Aunque a nosotros no nos pone un `Crawl-delay`, el sitio sí se lo pide a otros bots automáticos: eso muestra que gob.pe espera que los programas no lo saturen con pedidos seguidos. Es un servidor público que usan los ciudadanos; si hacemos muchos pedidos sin pausa podemos cargarlo, y además el sitio podría bloquearnos. Por eso esperamos **2 segundos** entre páginas de búsqueda (Selenium) y **1 segundo** entre páginas de normas (`requests`).

## 2–4. Selenium: un solo mes

Abrimos la búsqueda de **febrero de 2024** para probar. Usamos `WebDriverWait` para esperar a que la página termine de cargar los resultados con JavaScript (en lugar de un `time.sleep()` fijo, que puede quedarse corto o esperar de más).

Qué esperamos exactamente:
1. Que aparezca el contador `<span aria-label="N resultados encontrados">` → de ahí leemos **N** (paso 3).
2. Que en la página haya **al menos N** `<article>`, para no leer la lista a medio cargar.

In [3]:
URL_BASE = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
            "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")

opciones = webdriver.ChromeOptions()
opciones.add_argument("--headless=new")   # sin abrir la ventana de Chrome
driver = webdriver.Chrome(options=opciones)

In [4]:
def buscar_mes(driver, anio, mes):
    """Abre la búsqueda de un mes y devuelve (total_que_muestra_la_pagina, lista_de_resultados)."""
    ultimo_dia = calendar.monthrange(anio, mes)[1]          # 29 en febrero de 2024 (bisiesto)
    desde = f"01-{mes:02d}-{anio}"
    hasta = f"{ultimo_dia:02d}-{mes:02d}-{anio}"
    driver.get(f"{URL_BASE}&desde={desde}&hasta={hasta}")

    espera = WebDriverWait(driver, 20)
    # Paso 3: número total de resultados ("N Resultados")
    contador = espera.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "span[aria-label$='resultados encontrados']")))
    total = int(re.search(r"\d+", contador.text).group())

    # Esperamos a que estén cargados todos los <article>
    espera.until(lambda d: len(d.find_elements(By.TAG_NAME, "article")) >= total)

    # Paso 4: dentro de cada <article> buscamos cada dato
    resultados = []
    for articulo in driver.find_elements(By.TAG_NAME, "article"):
        enlace = articulo.find_element(By.CSS_SELECTOR, "a.link-principal")
        fecha = articulo.find_element(By.XPATH, ".//span[starts-with(normalize-space(), 'Publicado:')]")
        titulo = articulo.find_element(By.CSS_SELECTOR, "p")   # el párrafo debajo de la fecha
        resultados.append({
            "numero": enlace.text.strip(),
            "fecha": fecha.text.replace("Publicado:", "").strip(),
            "titulo_corto": titulo.text.strip(),
            "enlace": enlace.get_attribute("href"),   # Selenium devuelve la URL completa
        })
    return total, resultados

total_feb, resultados_feb = buscar_mes(driver, 2024, 2)
print("Total que muestra la página:", total_feb)
print("Resultados extraídos:", len(resultados_feb))
pd.DataFrame(resultados_feb).head()

Total que muestra la página: 11
Resultados extraídos: 11


,numero,fecha,titulo_corto,enlace
0,Decreto Supremo N.° 021-2024-PCM,28 de febrero de 2024,Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos d...,https://www.gob.pe/institucion/pcm/normas-legales/5433283-021-2024-pcm
1,Decreto Supremo N.° 020-2024-PCM,26 de febrero de 2024,Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos d...,https://www.gob.pe/institucion/pcm/normas-legales/5533513-020-2024-pcm
2,Resolución N.° 0634-2024-TCE-S5,23 de febrero de 2024,"Recurso de apelación interpuesto por la empresa SIMED PERU S.A.C., en la ADJUDICACIÓN SIMPLIFICADA Nº 017-2023-HEAV-...",https://www.gob.pe/institucion/oece/normas-legales/5260816-0634-2024-tce-s5
3,Resolución N.° 0533-2024-TCE-S4,14 de febrero de 2024,"Recurso de apelación interpuesto por la empresa DIAGNOSTICA PERUANA S.A.C., en el marco de la Licitación Pública Nº ...",https://www.gob.pe/institucion/oece/normas-legales/5197999-0533-2024-tce-s4
4,Decreto Supremo N.° 019-2024-PCM,13 de febrero de 2024,Decreto Supremo que declara el Estado de Emergencia en la provincia de Trujillo del departamento de La Libertad,https://www.gob.pe/institucion/pcm/normas-legales/5533472-019-2024-pcm


## 5–6. Todos los meses de la temporada + verificación

Repetimos para enero–mayo de 2024 con una pausa de 2 segundos entre páginas. Luego comparamos, mes por mes, el total que dice la página con lo que realmente extrajimos.

In [5]:
ANIO = 2024
MESES = [1, 2, 3, 4, 5]

todas = []
verificacion = []
for mes in MESES:
    total, resultados = buscar_mes(driver, ANIO, mes)
    for r in resultados:
        r["mes"] = mes
    todas.extend(resultados)
    verificacion.append({"mes": mes, "resultados_totales": total,
                         "resultados_extraidos": len(resultados)})
    time.sleep(2)   # pausa entre página y página

driver.quit()

verificacion = pd.DataFrame(verificacion)
verificacion["coinciden"] = verificacion["resultados_totales"] == verificacion["resultados_extraidos"]
print(verificacion.to_string(index=False))
assert verificacion["coinciden"].all(), "¡Hay meses donde no coinciden! Revisar antes de seguir."

 mes  resultados_totales  resultados_extraidos  coinciden
   1                  11                    11       True
   2                  11                    11       True
   3                  11                    11       True
   4                  15                    15       True
   5                   3                     3       True


Los dos números coinciden en todos los meses, así que la extracción está completa. (Ningún mes de 2024 tiene más resultados de los que caben en una página, por eso no tuvimos que recorrer paginación; si hubiera sido así, `resultados_extraidos` habría salido menor y el `assert` lo habría detectado.)

## 7. Eliminar normas repetidas

In [6]:
normas = pd.DataFrame(todas)
antes = len(normas)
# Una misma norma tiene siempre el mismo enlace: lo usamos como identificador.
normas = normas.drop_duplicates(subset="enlace").reset_index(drop=True)
print(f"Filas antes: {antes} | después: {len(normas)} | repetidas eliminadas: {antes - len(normas)}")

Filas antes: 51 | después: 51 | repetidas eliminadas: 0


## 8. Quedarnos solo con normas de la PCM

In [7]:
es_pcm = normas["enlace"].str.contains("/institucion/pcm/", regex=False)
print("Normas de otras instituciones que quitamos:", (~es_pcm).sum())
print(normas.loc[~es_pcm, ["numero", "enlace"]].to_string(index=False))
normas = normas[es_pcm].reset_index(drop=True)
print("\nNormas de la PCM que quedan:", len(normas))

Normas de otras instituciones que quitamos: 8
                         numero                                                                      enlace
Resolución N.° 0214-2024-TCE-S6 https://www.gob.pe/institucion/oece/normas-legales/5187307-0214-2024-tce-s6
Resolución N.° 0634-2024-TCE-S5 https://www.gob.pe/institucion/oece/normas-legales/5260816-0634-2024-tce-s5
Resolución N.° 0533-2024-TCE-S4 https://www.gob.pe/institucion/oece/normas-legales/5197999-0533-2024-tce-s4
Resolución N.° 0506-2024-TCE-S5 https://www.gob.pe/institucion/oece/normas-legales/5193883-0506-2024-tce-s5
Resolución N.° 0437-2024-TCE-S3 https://www.gob.pe/institucion/oece/normas-legales/5192380-0437-2024-tce-s3
Resolución N.° 0991-2024-TCE-S6 https://www.gob.pe/institucion/oece/normas-legales/5421041-0991-2024-tce-s6
Resolución N.° 0972-2024-TCE-S2 https://www.gob.pe/institucion/oece/normas-legales/5420398-0972-2024-tce-s2
Resolución N.° 1071-2024-TCE-S4 https://www.gob.pe/institucion/oece/normas-legales/5464861

## 9. Título completo con `requests` + `BeautifulSoup`

La página de cada norma no necesita JavaScript. El título completo está en `<div class="description">`; al final trae un texto extra (*"DS N° ... PDF 1.4 MB Descargar"*) que **limpiamos** para que no ensucie la búsqueda de palabras en los pasos siguientes.

Primero intentamos cortarlo con una expresión regular desde `" DS N"`, pero al revisar los resultados vimos que ese texto **no siempre está escrito igual** (a veces dice `Decreto Supremo N° ...`, otras `DS 016-...` sin "N°", y en algunas quedaba un `1` suelto). Revisando el HTML vimos que todo ese texto extra está dentro de un `<div class="institution-document__files">` (la tarjeta del PDF). Así que lo más seguro es **borrar ese div** con `.decompose()` antes de sacar el texto.

In [8]:
def titulo_completo(enlace):
    respuesta = requests.get(enlace, headers=HEADERS, timeout=30)
    respuesta.raise_for_status()
    sopa = BeautifulSoup(respuesta.text, "html.parser")
    descripcion = sopa.find("div", class_="description")
    # Quitamos la tarjeta del PDF ("DS N° 021-2024-PCM PDF 1.4 MB Descargar")
    for tarjeta in descripcion.find_all("div", class_="institution-document__files"):
        tarjeta.decompose()
    return descripcion.get_text(" ", strip=True)

titulos = []
for enlace in normas["enlace"]:
    titulos.append(titulo_completo(enlace))
    time.sleep(1)   # pausa entre páginas
normas["titulo"] = titulos

# Comprobaciones: ningún título debe seguir cortado ni traer la cola del PDF
print("Títulos que siguen cortados:", normas["titulo"].str.endswith("...").sum())
print("Títulos con 'Descargar':", normas["titulo"].str.contains("Descargar").sum())
normas[["numero", "titulo_corto", "titulo"]].head(3)

Títulos que siguen cortados: 0
Títulos con 'Descargar': 0


,numero,titulo_corto,titulo
0,Decreto Supremo N.° 010-2024-PCM,"Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado ...","Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado ..."
1,Decreto Supremo N.° 009-2024-PCM,"Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia...","Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia..."
2,Decreto Supremo N.° 008-2024-PCM,"Declarar por el termino de treinta (30) días calendario, a partir del 5 de febrero de 2024 el Estado de Emergencia e...","Declarar por el termino de treinta (30) días calendario, a partir del 5 de febrero de 2024 el Estado de Emergencia e..."


## 10. Columnas `es_lluvia`, `tipo` y `motivo`

In [9]:
def es_lluvia(titulo):
    t = titulo.lower()
    return "lluvia" in t or "precipitaciones" in t

def tipo(titulo):
    t = titulo.lower()
    if "prórroga" in t or "prorroga" in t:
        return "prorroga"
    elif "declara" in t:
        return "declara"
    else:
        return "otro"

def motivo(titulo):
    t = titulo.lower()
    if "peligro inminente" in t:
        return "peligro inminente"
    elif "impacto de daños" in t:
        return "impacto de daños"
    else:
        return "otro"

normas["es_lluvia"] = normas["titulo"].apply(es_lluvia)
normas["tipo"] = normas["titulo"].apply(tipo)
normas["motivo"] = normas["titulo"].apply(motivo)

print(normas["es_lluvia"].value_counts(), "\n")
print(pd.crosstab(normas["tipo"], normas["motivo"], margins=True))

es_lluvia
False    23
True     20
Name: count, dtype: int64 

motivo    impacto de daños  otro  peligro inminente  All
tipo                                                    
declara                 12     5                  0   17
prorroga                 8    16                  2   26
All                     20    21                  2   43


Revisamos las filas que quedaron como `"otro"` en `tipo` o en `motivo`:

In [10]:
otros = normas[(normas["tipo"] == "otro") | (normas["motivo"] == "otro")]
for _, fila in otros.iterrows():
    print(f"[{fila['numero']}] es_lluvia={fila['es_lluvia']} | tipo={fila['tipo']} | motivo={fila['motivo']}")
    print("   ", fila["titulo"], "\n")

[Decreto Supremo N.° 010-2024-PCM] es_lluvia=False | tipo=prorroga | motivo=otro
    Prorrogar por el termino de sesenta (60) días calendario a partir del 6 de febrero al 5 de abril de 2024, el Estado de Emergencia en los distritos que se indican en el Anexo que forma parte integrante del presente Decreto Supremo. 

[Decreto Supremo N.° 009-2024-PCM] es_lluvia=False | tipo=prorroga | motivo=otro
    Prorrogar por el termino de sesenta (60) días calendario, a partir del 1 de febrero de 2024, el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios. La Policía Nacional del Perú mantiene el control del orden interno, con el apoyo de las Fuerzas Armadas. 

[Decreto Supremo N.° 008-2024-PCM] es_lluvia=False | tipo=declara | motivo=otro
    Declarar por el termino de treinta (30) días calendario, a partir del 5 de feb

También revisamos al revés: normas que **no** quedaron como lluvia pero sí tienen un motivo ("peligro inminente" o "impacto de daños"), por si el filtro de lluvia se nos escapó alguna:

In [11]:
sospechosas = normas[~normas["es_lluvia"] & (normas["motivo"] != "otro")]
for _, fila in sospechosas.iterrows():
    print(f"[{fila['numero']}] tipo={fila['tipo']} | motivo={fila['motivo']}")
    print("   ", fila["titulo"], "\n")

[Decreto Supremo N.° 052-2024-PCM] tipo=declara | motivo=impacto de daños
    Decreto Supremo que declara el Estado de Emergencia en los distritos Pachacamac y Villa María del Triunfo de la provincia de Lima del departamento de Lima, por impacto de daños a consecuencia de explosión en estación de carga de gas natural comprimido 

[Decreto Supremo N.° 050-2024-PCM] tipo=declara | motivo=impacto de daños
    Decreto Supremo que declara el Estado de Emergencia en varios distritos de la provincia de Quispicanchi del departamento de Cusco, por impacto de daños ante la ocurrencia de movimiento sísmico 



**¿Están bien clasificadas las filas "otro"?**

- **`tipo`**: ninguna norma quedó como `"otro"`. Todas dicen *declara/Declarar/Declaratoria* o *prorroga/Prorrogar/Prórroga*. Un detalle importante: los títulos de prórroga dicen *"prorroga el Estado de Emergencia **declarado**..."*, que también contiene `"declara"`. Por eso en la función **preguntamos primero por "prórroga"** y solo después por "declara"; si fuera al revés, todas las prórrogas saldrían como declaratorias.
- **`motivo`**: las 21 normas con `motivo = "otro"` tienen `es_lluvia = False`, y está bien que sea así: son estados de emergencia por **otras razones** (criminalidad u orden interno, por ejemplo en Pataz, Trujillo, el Corredor Vial Sur o Putumayo, donde el título dice *"La Policía Nacional del Perú mantiene el control del orden interno"*), y su título no menciona "peligro inminente" ni "impacto de daños". No se nos escapa ninguna norma de lluvias: **las 20 normas de lluvia tienen motivo asignado** (18 por impacto de daños y 2 por peligro inminente).
- Caso a mencionar: el **DS 010-2024-PCM** solo dice *"en los distritos que se indican en el Anexo"*. Por el título no podemos saber el motivo ni si es por lluvias, así que queda fuera. Es una limitación: solo miramos el título, no el PDF.
- Revisión al revés: **DS 052-2024-PCM** (explosión en una estación de gas natural) y **DS 050-2024-PCM** (sismo en Quispicanchi) dicen "impacto de daños", pero **no son por lluvias**. Por eso está bien que `es_lluvia` sea `False`: el motivo por sí solo no basta para decidir si una norma es de lluvias.
- El **DS 0012-2024-PCM** tiene una errata en la fuente (*"**pro** impacto de daños"*), pero igual contiene `"impacto de daños"` y queda bien clasificado.

## 11. Solo normas de lluvias

In [12]:
lluvias = normas[normas["es_lluvia"]].reset_index(drop=True)
print("Normas de lluvias:", len(lluvias))
lluvias[["numero", "fecha", "tipo", "motivo", "titulo"]].head(5)

Normas de lluvias: 20


,numero,fecha,tipo,motivo,titulo
0,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,prorroga,impacto de daños,Decreto Supremo que prorroga el Estado de Emergencia en el distrito de Echarate de la provincia de La Convención del...
1,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,prorroga,peligro inminente,Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos ...
2,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos d...
3,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emergencia en el distrito de Chalhuanca de la provincia de Aymaraes del dep...
4,Decreto Supremo N.° 021-2024-PCM,28 de febrero de 2024,declara,impacto de daños,Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos d...


## 12. Departamentos que menciona cada decreto

Decisiones:

1. **Buscamos en el título original (con mayúsculas) y por palabra completa** con `\b`: así `"Ica"` no coincide dentro de `"Huancavelica"` (ahí no hay un límite de palabra antes de `ica`, y además está en minúscula).
2. **Ignoramos solo las tildes** (no las mayúsculas): normalizamos título y departamento quitando tildes, para que `"Ancash"` o `"Junin"` escritos sin tilde también cuenten. Las mayúsculas se mantienen, que es lo que evita el problema de `huancavelica`.
3. **Contamos cada departamento una sola vez por decreto**: por cada departamento de la lista preguntamos *si aparece* (sí/no) en el título, no *cuántas veces* aparece. Así, *"provincia de Ica del departamento de Ica"* suma 1 para Ica, no 2.

In [13]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]

def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto)
                   if unicodedata.category(c) != "Mn")

def departamentos_en(titulo):
    t = sin_tildes(titulo)
    # Una sola entrada por departamento (re.search dice si aparece o no, no cuántas veces)
    return [d for d in departamentos if re.search(r"\b" + sin_tildes(d) + r"\b", t)]

lluvias["departamentos"] = lluvias["titulo"].apply(departamentos_en)
lluvias["n_departamentos"] = lluvias["departamentos"].str.len()

print("Decretos sin ningún departamento detectado:", (lluvias["n_departamentos"] == 0).sum())
lluvias[["numero", "tipo", "departamentos"]]

Decretos sin ningún departamento detectado: 0


,numero,tipo,departamentos
0,Decreto Supremo N.° 007-2024-PCM,prorroga,[Cusco]
1,Decreto Supremo N.° 006-2024-PCM,prorroga,"[Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, Lima, Pasco, Piura, San Martín, Tumbes]"
2,Decreto Supremo N.° 005-2024-PCM,declara,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Ica, Moquegua, Pasco]"
3,Decreto Supremo N.° 004-2024-PCM,declara,[Apurímac]
4,Decreto Supremo N.° 021-2024-PCM,declara,"[Arequipa, Ica, Madre de Dios, Moquegua, Ucayali]"
5,Decreto Supremo N.° 020-2024-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, L..."
6,Decreto Supremo N.° 016-2024-PCM,declara,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Junín, Lima, Madre de Dios, Pasco, Pu..."
7,Decreto Supremo N.° 0012-2024-PCM,declara,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, Loreto, San Martín]"
8,Decreto Supremo N.° 033-2024-PCM,prorroga,"[Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Ica, Moquegua, Pasco]"
9,Decreto Supremo N.° 032-2024-PCM,prorroga,"[Áncash, Apurímac, Cusco, Huancavelica, Lima, Loreto, San Martín]"


Comprobamos el caso de Ica / Huancavelica en nuestra temporada:

In [14]:
# ¿Cuántas veces aparece la palabra "Ica" en cada título que la menciona?
lluvias["veces_Ica"] = lluvias["titulo"].apply(lambda t: len(re.findall(r"\bIca\b", t)))
lluvias["menciona_Huancavelica"] = lluvias["titulo"].str.contains("Huancavelica")
lluvias["ica_en_minusculas"] = lluvias["titulo"].str.lower().str.contains("ica")   # la forma INCORRECTA

casos = lluvias[lluvias["menciona_Huancavelica"] | (lluvias["veces_Ica"] > 0)]
for _, fila in casos.iterrows():
    print(f"[{fila['numero']}] 'Ica' aparece {fila['veces_Ica']} vez/veces | "
          f"Huancavelica={fila['menciona_Huancavelica']} | "
          f"'ica' en minúsculas={fila['ica_en_minusculas']} | "
          f"¿Ica en nuestra lista? {'Ica' in fila['departamentos']}")
    print("   ", fila["titulo"], "\n")

print("Con la búsqueda INCORRECTA ('ica' in titulo.lower()), Ica tendría",
      lluvias["ica_en_minusculas"].sum(), "decretos.")
print("Con nuestra búsqueda, Ica tiene", lluvias["departamentos"].apply(lambda x: "Ica" in x).sum(), "decretos.")
lluvias = lluvias.drop(columns=["veces_Ica", "menciona_Huancavelica", "ica_en_minusculas"])

[Decreto Supremo N.° 006-2024-PCM] 'Ica' aparece 1 vez/veces | Huancavelica=False | 'ica' en minúsculas=True | ¿Ica en nuestra lista? True
    Decreto Supremo que prorroga el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Ica, La Libertad, Lambayeque, Lima, Pasco, Piura, San Martín y Tumbes, por peligro inminente ante intensas precipitaciones pluviales (período 2023- 2024) y posible Fenómeno El Niño 

[Decreto Supremo N.° 005-2024-PCM] 'Ica' aparece 1 vez/veces | Huancavelica=True | 'ica' en minúsculas=True | ¿Ica en nuestra lista? True
    Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Arequipa, Ayacucho, Cusco, Huánuco, Huancavelica, Ica, Moquegua y Pasco, por impacto de daños a consecuencia de intensas precipitaciones pluviales 

[Decreto Supremo N.° 021-2024-PCM] 'Ica' aparece 1 vez/veces | Huancavelica=False | 'ica' en minú

**¿Cómo evitamos contar mal? Un ejemplo de 2024**

- **Huancavelica ≠ Ica.** El **DS 016-2024-PCM** declara emergencia en *"...Ayacucho, Cusco, **Huancavelica**, Huánuco, Junín..."* y **no menciona Ica**. Si buscáramos `"ica" in titulo.lower()`, este decreto contaría para Ica, porque *huancavel**ica*** contiene `"ica"`. En nuestra temporada, esa búsqueda incorrecta le daría a Ica **14 decretos**, cuando en realidad tiene **7**: la mitad serían falsos. Con `re.search(r"\bIca\b", titulo)` sobre el título original, *Huancavelica* no coincide: dentro de esa palabra no hay límite de palabra (`\b`) antes de `ica` y además la `i` está en minúscula.
- **Una sola vez por decreto.** En los títulos de 2024 no apareció el caso *"provincia de Ica del departamento de Ica"* (en la tabla de arriba, "Ica" aparece como máximo 1 vez por título), pero el código ya lo cubre: recorremos la lista de 25 departamentos y para cada uno preguntamos **si aparece o no** (`re.search`), en vez de contar cuántas veces aparece (`re.findall`). Cada departamento entra como máximo una vez en la lista de cada decreto, aunque el nombre se repita.
- **Tildes.** El **DS 0012-2024-PCM** escribe *"**Ancash**"* y *"San **Martin**"* sin tilde. Con una búsqueda exacta de `"Áncash"` y `"San Martín"` esos dos departamentos habrían perdido una declaratoria cada uno. Por eso quitamos las tildes (del título y del nombre) antes de comparar, pero **mantenemos las mayúsculas**.

## 13. Guardar los archivos

In [15]:
lluvias[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]].to_csv(
    CARPETA_DATOS / "decretos_lluvias.csv", index=False, encoding="utf-8-sig")

# Una fila por (decreto, departamento) y contamos declaratorias y prórrogas por departamento
por_depto = (lluvias.explode("departamentos", ignore_index=True)
                    .rename(columns={"departamentos": "departamento"}))
conteo = (pd.crosstab(por_depto["departamento"], por_depto["tipo"])
            .reindex(departamentos, fill_value=0))     # los 25, con 0 si no tuvieron ninguno
for col in ["declara", "prorroga"]:
    if col not in conteo:
        conteo[col] = 0
decretos_por_departamento = (conteo.rename(columns={"declara": "declaratorias", "prorroga": "prorrogas"})
                             [["declaratorias", "prorrogas"]]
                             .rename_axis("departamento")
                             .reset_index())
decretos_por_departamento.columns.name = None
decretos_por_departamento.to_csv(CARPETA_DATOS / "decretos_por_departamento.csv",
                                 index=False, encoding="utf-8-sig")

print(decretos_por_departamento.sort_values("declaratorias", ascending=False).to_string(index=False))

 departamento  declaratorias  prorrogas
     Apurímac              7          4
        Cusco              7          5
     Arequipa              7          6
     Ayacucho              7          5
 Huancavelica              7          4
         Lima              6          5
      Huánuco              6          3
       Áncash              5          5
        Junín              5          1
         Puno              5          2
     Moquegua              5          2
     Amazonas              4          2
      Ucayali              4          0
        Tacna              4          0
          Ica              4          3
   San Martín              3          4
    Cajamarca              3          3
       Loreto              3          2
        Pasco              3          3
Madre de Dios              3          0
  La Libertad              2          3
       Callao              0          0
   Lambayeque              0          2
        Piura              0          2


In [16]:
# Verificamos que los archivos se leen bien
print(pd.read_csv(CARPETA_DATOS / "decretos_lluvias.csv").shape)
print(pd.read_csv(CARPETA_DATOS / "decretos_por_departamento.csv").shape)

(20, 6)
(25, 3)
